In [63]:
# ============================================================
#安装依赖
# ============================================================
!pip install h5py scikit-learn matplotlib seaborn torch -q
print('✅ 安装完成')

✅ 安装完成


In [64]:
# ============================================================
# Cell 1: 安装与导入依赖
# ============================================================

# Kaggle 默认一般有 torch / scipy / sklearn / h5py
# 但 pyts 通常没有，需要单独安装
!pip install pyts -q

import os
import json
import h5py
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from collections import defaultdict

from scipy import signal
from scipy.ndimage import zoom

# GASF / GADF / RP 需要用到 pyts
from pyts.image import GramianAngularField
from pyts.image import RecurrencePlot

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    f1_score,
    roc_curve,
    auc,
    precision_recall_curve,
    average_precision_score
)

import warnings
warnings.filterwarnings("ignore")

# 固定随机种子，保证实验尽量可复现
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

# 自动选择 GPU 或 CPU
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"✅ 使用设备: {DEVICE}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

print("✅ 依赖安装与导入完成")

✅ 使用设备: cuda
GPU: Tesla T4
✅ 依赖安装与导入完成


In [65]:
# ============================================================
# Cell 3: 分层均匀抽样
#
# 只读 Y（标签）和 Z（SNR），建立抽样索引。
# X 数据不在这里读取！
#
# 抽样策略：
#   每种调制方式：10,000 条
#   每种调制内按 SNR 均匀分配：10000 / 26档 ≈ 384条/档
#   总计：24 × 10000 = 240,000 条
# ============================================================

HDF5_PATH = '/kaggle/input/datasets/pinxau1000/radioml2018/GOLD_XYZ_OSC.0001_1024.hdf5'

# ── 步骤1：只读 Y 和 Z（内存友好，几十MB）────────────────────
print('📖 读取 Y 和 Z...')
with h5py.File(HDF5_PATH, 'r') as f:
    Y_raw = f['Y'][:]   # (N, 24)  One-hot，几十MB
    Z_raw = f['Z'][:]   # (N, 1)   SNR，几MB

y_mod_full = np.argmax(Y_raw, axis=1)   # (N,) 调制类型索引
snr_full   = Z_raw.flatten()             # (N,) SNR值
del Y_raw, Z_raw   # 立即释放原始one-hot数组

MOD_CLASSES = [
    'OOK','4ASK','8ASK','BPSK','QPSK','8PSK',
    '16PSK','32PSK','16APSK','32APSK','64APSK',
    '128APSK','16QAM','32QAM','64QAM','128QAM',
    '256QAM','AM-SSB-WC','AM-SSB-SC','AM-DSB-WC',
    'AM-DSB-SC','FM','GMSK','OQPSK'
]
NUM_MOD  = len(MOD_CLASSES)
ALL_SNRS = sorted(np.unique(snr_full).tolist())   # [-20,-18,...,+30]
ALL_MODS = sorted(np.unique(y_mod_full).tolist()) #是一个列表

print(f'  总样本数: {len(y_mod_full):,}')
print(f'  调制种类: {NUM_MOD}  |  SNR档位: {len(ALL_SNRS)}')
print(f'  SNR范围: {ALL_SNRS[0]} ~ {ALL_SNRS[-1]} dB')

# ── 步骤2：分层均匀抽样，只建索引 ───────────────────────────
SAMPLES_PER_MOD = 10_000                            # 每种调制总条数
SAMPLES_PER_SNR = SAMPLES_PER_MOD // len(ALL_SNRS) # 每档SNR的条数（约384）

print(f'\n🎯 抽样计划：')
print(f'  每种调制: {SAMPLES_PER_MOD:,} 条')
print(f'  每档SNR : {SAMPLES_PER_SNR} 条')
print(f'  总计    : {NUM_MOD * SAMPLES_PER_MOD:,} 条')
print('\n计算索引中...')

selected_indices = []   # 只存整数索引，约 240000×4字节 ≈ 1MB，极省内存

for m in ALL_MODS:
    for s in ALL_SNRS:
        # 找到 调制=m 且 SNR=s 的所有行号
        pool = np.where((y_mod_full == m) & (snr_full == s))[0]
        n_take = min(SAMPLES_PER_SNR, len(pool))
        chosen = np.random.choice(pool, n_take, replace=False)
        selected_indices.extend(chosen.tolist())

# 排序！HDF5顺序读取比随机跳读快 10~100 倍
selected_indices = sorted(selected_indices)
selected_indices = np.array(selected_indices, dtype=np.int32)

# 对应的调制类型和SNR（从全量数组直接索引，无需读X）
y_mod_sel = y_mod_full[selected_indices]   # (240000,)
snr_sel   = snr_full[selected_indices]     # (240000,)

del y_mod_full, snr_full   # 释放全量数组

# ── 步骤3：验证抽样均匀性 ────────────────────────────────────
print(f'\n✅ 抽样完成！总索引数: {len(selected_indices):,}')
print(f'\n每种调制的样本数（应均为 {SAMPLES_PER_MOD}）：')
for m_idx in ALL_MODS:
    cnt = (y_mod_sel == m_idx).sum()
    bar = '█' * (cnt // 200)
    print(f'  [{m_idx:2d}] {MOD_CLASSES[m_idx]:<12} {cnt:5,} {bar}')

print(f'\n每档SNR的样本数（应均匀）：')
for s in ALL_SNRS:
    cnt = (snr_sel == s).sum()
    print(f'  SNR={s:4.0f}dB  {cnt:,}')

📖 读取 Y 和 Z...
  总样本数: 2,555,904
  调制种类: 24  |  SNR档位: 26
  SNR范围: -20 ~ 30 dB

🎯 抽样计划：
  每种调制: 10,000 条
  每档SNR : 384 条
  总计    : 240,000 条

计算索引中...

✅ 抽样完成！总索引数: 239,616

每种调制的样本数（应均为 10000）：
  [ 0] OOK          9,984 █████████████████████████████████████████████████
  [ 1] 4ASK         9,984 █████████████████████████████████████████████████
  [ 2] 8ASK         9,984 █████████████████████████████████████████████████
  [ 3] BPSK         9,984 █████████████████████████████████████████████████
  [ 4] QPSK         9,984 █████████████████████████████████████████████████
  [ 5] 8PSK         9,984 █████████████████████████████████████████████████
  [ 6] 16PSK        9,984 █████████████████████████████████████████████████
  [ 7] 32PSK        9,984 █████████████████████████████████████████████████
  [ 8] 16APSK       9,984 █████████████████████████████████████████████████
  [ 9] 32APSK       9,984 █████████████████████████████████████████████████
  [10] 64APSK       9,984 ████████████████████

In [66]:
# ============================================================
# Cell 4: 定义已知/未知调制，以及各数据集的SNR区间
# ============================================================

# ── 调制方式划分 ─────────────────────────────────────────────
KNOWN_MODS = [
    'BPSK','QPSK','8PSK','16PSK',          # 相位调制族
    'OOK','4ASK','8ASK',                    # 幅度调制族
    '16QAM','32QAM','64QAM',                # QAM族（低阶）
    'AM-DSB-SC','AM-SSB-SC',               # 模拟调制（抑制载波）
    'GMSK','OQPSK'                          # 扩频/特殊
]
UNKNOWN_MODS = [
    '32PSK','16APSK','32APSK','64APSK','128APSK',   # 高阶相位/幅度相位
    '128QAM','256QAM',                               # 高阶QAM
    'AM-DSB-WC','AM-SSB-WC',                         # 带载波模拟调制
    'FM'                                              # 频率调制
]

known_idx   = [MOD_CLASSES.index(m) for m in KNOWN_MODS]
unknown_idx = [MOD_CLASSES.index(m) for m in UNKNOWN_MODS]
assert len(known_idx) + len(unknown_idx) == NUM_MOD

# ── SNR 区间规划 ──────────────────────────────────────────────
#
# 训练集 H1：中间区间 [-10, +10] dB（11档）
#   模型见过这个SNR范围，但不含极端情况
#
# 测试集1 H1：训练未覆盖的SNR
#   低端：[-18, -12] dB（低SNR外推）
#   高端：[+12, +30] dB（高SNR外推）
#   目的：测试SNR泛化能力
#
# 测试集2 H1：全SNR范围（排除-20dB）
#   目的：测试调制泛化能力（盲检测）
#
# H0：SNR=-20dB 的样本（信号被噪声淹没，工程上≈空闲）
#     + 纯AWGN（增强H0多样性）

H0_SNR        = -20
TRAIN_H1_SNRS = [s for s in ALL_SNRS if -15 <= s <= 15]
TEST1_H1_SNRS = [s for s in ALL_SNRS if (s < -15 or s > 15) and s != H0_SNR]
TEST2_H1_SNRS = [s for s in ALL_SNRS if s != H0_SNR]

print('='*60)
print('📋 实验设计确认')
print('='*60)
print(f'已知调制（{len(KNOWN_MODS)}种）: {KNOWN_MODS}')
print(f'未知调制（{len(UNKNOWN_MODS)}种）: {UNKNOWN_MODS}')
print(f'训练H1 SNR ({len(TRAIN_H1_SNRS)}档): {TRAIN_H1_SNRS}')
print(f'测试1 H1 SNR({len(TEST1_H1_SNRS)}档): {TEST1_H1_SNRS}')
print(f'测试2 H1 SNR({len(TEST2_H1_SNRS)}档): {TEST2_H1_SNRS}')
print(f'H0 SNR: {H0_SNR} dB')

📋 实验设计确认
已知调制（14种）: ['BPSK', 'QPSK', '8PSK', '16PSK', 'OOK', '4ASK', '8ASK', '16QAM', '32QAM', '64QAM', 'AM-DSB-SC', 'AM-SSB-SC', 'GMSK', 'OQPSK']
未知调制（10种）: ['32PSK', '16APSK', '32APSK', '64APSK', '128APSK', '128QAM', '256QAM', 'AM-DSB-WC', 'AM-SSB-WC', 'FM']
训练H1 SNR (15档): [-14, -12, -10, -8, -6, -4, -2, 0, 2, 4, 6, 8, 10, 12, 14]
测试1 H1 SNR(10档): [-18, -16, 16, 18, 20, 22, 24, 26, 28, 30]
测试2 H1 SNR(25档): [-18, -16, -14, -12, -10, -8, -6, -4, -2, 0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30]
H0 SNR: -20 dB


In [67]:
# ============================================================
# Cell 5: 构建三阶段数据集索引（修正版）
# ============================================================

def get_mask(y_arr, snr_arr, mod_indices, snr_list):
    return np.isin(y_arr, mod_indices) & np.isin(snr_arr, snr_list)

def generate_awgn(n_samples, snr_db_list, n_points=1024):
    """
    生成纯AWGN负样本。
    σ = 1/√(2×10^(γ/10))，I,Q ~ N(0,σ²)
    shape: (n_samples, 1024, 2)
    """
    per_snr = max(1, n_samples // len(snr_db_list))
    chunks  = []
    #遍历每个SNR -20、-18、、、20、、、
    for snr_db in snr_db_list:
        snr_lin = 10 ** (snr_db / 10.0)#对数转换成线性
        sigma   = 1.0 / np.sqrt(2.0 * snr_lin)#对应公式AWGN生成公式
        noise   = (np.random.randn(per_snr, n_points, 2) * sigma).astype(np.float32)
        chunks.append(noise)
    arr = np.concatenate(chunks, axis=0)
    # 补足或截断到精确数量
    while len(arr) < n_samples:
        arr = np.concatenate([arr, arr], axis=0)
    return arr[:n_samples]

# ================================================================
# 训练+验证集
# ================================================================
print('📦 构建训练/验证集...')

# H1：14种已知调制，SNR∈[-10,+10]dB
mask_tv_h1 = get_mask(y_mod_sel, snr_sel, known_idx, TRAIN_H1_SNRS)
tv_h1_pos  = np.where(mask_tv_h1)[0]
n_h1_tv    = len(tv_h1_pos)

# H0来源1：已知调制，SNR=-20dB（数量有限，全部用上）
mask_tv_h0d = get_mask(y_mod_sel, snr_sel, known_idx, [H0_SNR])
tv_h0d_pos  = np.where(mask_tv_h0d)[0]
n_h0d_tv    = len(tv_h0d_pos)

# H0来源2：纯AWGN，补足到与H1等量
# 目标：H0总量 = H1总量（1:1均衡）
n_awgn_tv_need = n_h1_tv - n_h0d_tv   # AWGN补足剩余
n_awgn_tv_need = max(n_awgn_tv_need, n_h0d_tv)  # 至少和H0_data等量
awgn_tv = generate_awgn(n_awgn_tv_need, TRAIN_H1_SNRS)

# 最终H0总量
n_h0_tv = n_h0d_tv + len(awgn_tv)
# 让H1和H0总量对齐（取较小值）
n_final_tv = min(n_h1_tv, n_h0_tv)
tv_h1_pos  = tv_h1_pos[np.random.choice(n_h1_tv, n_final_tv, replace=False)]

print(f'  H1={len(tv_h1_pos):,}  H0_data={n_h0d_tv:,}  H0_awgn={len(awgn_tv):,}')
print(f'  H0总量={n_h0_tv:,}  最终H1={n_final_tv:,}')

# 80/20 划分（按比例切分每个来源）
def split80(arr, is_numpy_arr=False):
    n = len(arr)
    n_tr = int(0.8 * n)
    if is_numpy_arr:
        return arr[:n_tr], arr[n_tr:]
    return arr[:n_tr], arr[n_tr:]

train_h1_pos,  val_h1_pos  = split80(tv_h1_pos)
train_h0d_pos, val_h0d_pos = split80(tv_h0d_pos)
train_awgn,    val_awgn    = split80(awgn_tv, is_numpy_arr=True)

print(f'\n  训练集: H1={len(train_h1_pos):,}  H0_data={len(train_h0d_pos):,}  H0_awgn={len(train_awgn):,}')
print(f'  验证集: H1={len(val_h1_pos):,}  H0_data={len(val_h0d_pos):,}  H0_awgn={len(val_awgn):,}')
print(f'  训练集总量≈{len(train_h1_pos)+len(train_h0d_pos)+len(train_awgn):,}')

# ================================================================
# 测试集1（已知调制，训练外SNR → SNR泛化）
# ================================================================
print('\n📦 构建测试集1（SNR泛化）...')

mask_t1_h1 = get_mask(y_mod_sel, snr_sel, known_idx, TEST1_H1_SNRS)
t1_h1_pos  = np.where(mask_t1_h1)[0]
n_t1_h1    = len(t1_h1_pos)

# 测试集1的H0：只用AWGN（数量与H1相等，1:1）
awgn_t1   = generate_awgn(n_t1_h1, TEST1_H1_SNRS)
snr_t1_h1 = snr_sel[t1_h1_pos]   # 保存SNR信息，用于画Pd-SNR曲线

print(f'  H1={n_t1_h1:,}  H0_awgn={len(awgn_t1):,}')
print(f'  测试集1总量≈{n_t1_h1 + len(awgn_t1):,}')

# ================================================================
# 测试集2（未知调制，全SNR → 调制泛化）
# ================================================================
print('\n📦 构建测试集2（调制泛化）...')

mask_t2_h1  = get_mask(y_mod_sel, snr_sel, unknown_idx, TEST2_H1_SNRS)
t2_h1_pos   = np.where(mask_t2_h1)[0]
n_t2_h1     = len(t2_h1_pos)

# H0来源1：未知调制，SNR=-20dB
mask_t2_h0d = get_mask(y_mod_sel, snr_sel, unknown_idx, [H0_SNR])
t2_h0d_pos  = np.where(mask_t2_h0d)[0]
n_h0d_t2    = len(t2_h0d_pos)

# H0来源2：AWGN补足到与H1等量
n_awgn_t2_need = n_t2_h1 - n_h0d_t2
n_awgn_t2_need = max(n_awgn_t2_need, n_h0d_t2)
awgn_t2 = generate_awgn(n_awgn_t2_need, TEST2_H1_SNRS)

n_h0_t2  = n_h0d_t2 + len(awgn_t2)
n_t2_h1_final = min(n_t2_h1, n_h0_t2)
t2_h1_pos = t2_h1_pos[np.random.choice(n_t2_h1, n_t2_h1_final, replace=False)]

# 保存细粒度信息（用于热力图）
snr_t2_h1 = snr_sel[t2_h1_pos]
mod_t2_h1 = y_mod_sel[t2_h1_pos]

print(f'  H1={len(t2_h1_pos):,}  H0_data={n_h0d_t2:,}  H0_awgn={len(awgn_t2):,}')
print(f'  测试集2总量≈{len(t2_h1_pos)+n_h0d_t2+len(awgn_t2):,}')

print('\n✅ 索引构建完成！')
print(f'\n📊 各数据集规模汇总：')
print(f'  训练集 : ~{len(train_h1_pos)+len(train_h0d_pos)+len(train_awgn):,} 条')
print(f'  验证集 : ~{len(val_h1_pos)+len(val_h0d_pos)+len(val_awgn):,} 条')
print(f'  测试集1: ~{n_t1_h1+len(awgn_t1):,} 条  （SNR泛化）')
print(f'  测试集2: ~{len(t2_h1_pos)+n_h0d_t2+len(awgn_t2):,} 条  （调制泛化）')

📦 构建训练/验证集...
  H1=80,640  H0_data=5,376  H0_awgn=75,264
  H0总量=80,640  最终H1=80,640

  训练集: H1=64,512  H0_data=4,300  H0_awgn=60,211
  验证集: H1=16,128  H0_data=1,076  H0_awgn=15,053
  训练集总量≈129,023

📦 构建测试集1（SNR泛化）...
  H1=53,760  H0_awgn=53,760
  测试集1总量≈107,520

📦 构建测试集2（调制泛化）...
  H1=96,000  H0_data=3,840  H0_awgn=92,160
  测试集2总量≈192,000

✅ 索引构建完成！

📊 各数据集规模汇总：
  训练集 : ~129,023 条
  验证集 : ~32,257 条
  测试集1: ~107,520 条  （SNR泛化）
  测试集2: ~192,000 条  （调制泛化）


In [68]:
# ============================================================
# Cell 6: 内存友好型 Dataset
#
# 两种样本类型：
#   RadioML样本  → 通过 hdf5_positions 从文件读取
#   AWGN样本     → 直接存在 awgn_data numpy数组里
#
# __getitem__ 中完成：
#   1. 读取原始IQ数据（(2,1024) 或 (1024,2)）
#   2. 统一转为 (1024, 2)
#   3. 功率归一化
# ============================================================

#把 RadioML 里的真实 IQ 样本和你人工生成的 AWGN 噪声样本统一组织起来，
#供 DataLoader 按 batch 读取，用于训练 H0/H1 二分类模型。
class SpectrumDataset(Dataset):
    #hdf5_positions表示当前数据集中要使用哪些 RadioML 样本的位置hdf5_positions 不是原始 HDF5 文件里的真实行号，它只是筛选后数组里的位置。
    #hdf5_labels表示这些 HDF5 样本对应的 H0/H1 标签。
    #awgn_data表示人工生成的 AWGN 噪声样本。
    #selected_indices它表示你之前从完整 HDF5 文件中筛选出来的样本，在原始 HDF5 文件中的真实行号。selected_indices 是一个“映射表”：筛选后位置 → 原始 HDF5 行号
    def __init__(self, hdf5_path, hdf5_positions, hdf5_labels,
                 awgn_data, selected_indices):
        self.hdf5_path   = hdf5_path
        self.hdf5_rows   = selected_indices[hdf5_positions].astype(np.int32)
        self.hdf5_labels = np.array(hdf5_labels, dtype=np.int64)
        self.awgn_data   = awgn_data                              # (M, 1024, 2)
        self.awgn_labels = np.zeros(len(awgn_data), dtype=np.int64)
        self.n_hdf5      = len(self.hdf5_rows)
        self.n_awgn      = len(self.awgn_data)
        self.total       = self.n_hdf5 + self.n_awgn
        self._f          = None

    def _open(self):
        if self._f is None:
            self._f = h5py.File(self.hdf5_path, 'r')

    def __len__(self):
        return self.total
#当 DataLoader 需要第 idx 条样本时，Dataset 应该返回哪条数据和对应标签。
    def __getitem__(self, idx):
        if idx < self.n_hdf5:
            # RadioML样本：从HDF5按行读取
            self._open()
            row = int(self.hdf5_rows[idx])
            x   = self._f['X'][row]      # (2, 1024)
            # --- 强制形状修正逻辑 ---
            if x.shape == (2, 1024):
                x = x.transpose(1, 0)  # 确保转为 (1024, 2)
            elif x.shape != (1024, 2):
                # 应对可能的异常维度（如多出一维）
                print(x.shape)
                x = x.reshape(1024, 2)
            #x   = x.T                     # → (1024, 2)
            y   = int(self.hdf5_labels[idx])
        else:
            # AWGN样本：从内存数组取
            ai  = idx - self.n_hdf5
            x   = self.awgn_data[ai]     # (1024, 2)
            y   = int(self.awgn_labels[ai])

        # 功率归一化：x̂ = x / √(mean(I²+Q²) + ε)
        power = float(np.mean(x ** 2)) + 1e-8
        x     = (x / np.sqrt(power)).astype(np.float32)

        return torch.from_numpy(x), torch.tensor(y, dtype=torch.long)

    def __del__(self):
        if self._f is not None:
            try: self._f.close()
            except: pass


# ── 实例化三个数据集 ─────────────────────────────────────────

# 训练集（H1来自HDF5 + H0_data来自HDF5 + H0_awgn来自内存）
#训练集 = 真实 H1 信号 + 低 SNR 近似 H0 样本 + 纯 AWGN 噪声
train_ds = SpectrumDataset(
    hdf5_path        = HDF5_PATH,
    #它把两类来自 HDF5 的训练样本拼在一起 这个训练集里来自 HDF5 的部分，前面是 H1，后面是 H0_data。
    hdf5_positions   = np.concatenate([train_h1_pos, train_h0d_pos]),
    hdf5_labels      = np.concatenate([
                           np.ones(len(train_h1_pos), dtype=np.int64),
                           np.zeros(len(train_h0d_pos), dtype=np.int64)
                       ]),
    awgn_data        = train_awgn,
    selected_indices = selected_indices,
)

# 验证集
val_ds = SpectrumDataset(
    hdf5_path        = HDF5_PATH,
    hdf5_positions   = np.concatenate([val_h1_pos, val_h0d_pos]),
    hdf5_labels      = np.concatenate([
                           np.ones(len(val_h1_pos),  dtype=np.int64),
                           np.zeros(len(val_h0d_pos), dtype=np.int64)
                       ]),
    awgn_data        = val_awgn,
    selected_indices = selected_indices,
)

# 测试集1（H1来自HDF5 + H0全为AWGN）
test1_ds = SpectrumDataset(
    hdf5_path        = HDF5_PATH,
    hdf5_positions   = t1_h1_pos,
    hdf5_labels      = np.ones(len(t1_h1_pos), dtype=np.int64),
    awgn_data        = awgn_t1,
    selected_indices = selected_indices,
)

# 测试集2（H1来自HDF5 + H0_data来自HDF5 + H0_awgn来自内存）
test2_ds = SpectrumDataset(
    hdf5_path        = HDF5_PATH,
    hdf5_positions   = np.concatenate([t2_h1_pos, t2_h0d_pos]),
    hdf5_labels      = np.concatenate([
                           np.ones(len(t2_h1_pos),  dtype=np.int64),
                           np.zeros(len(t2_h0d_pos), dtype=np.int64)
                       ]),
    awgn_data        = awgn_t2,
    selected_indices = selected_indices,
)

BATCH_SIZE = 16   # 加大batch，提高HDF5读取效率

# num_workers=0 避免多进程下HDF5文件句柄冲突
# 如果Kaggle支持 fork，可以设为2
#Dataset 负责“告诉程序每一条样本怎么取”；
#DataLoader 负责“按 batch 一批一批取出来，并决定是否打乱、是否并行读取、是否加速拷贝到 GPU”。
train_loader = DataLoader(train_ds,  batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,    batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test1_loader = DataLoader(test1_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test2_loader = DataLoader(test2_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

print('✅ Dataset & DataLoader 构建完成')
print(f'  训练集: {len(train_ds):,}  验证集: {len(val_ds):,}')
print(f'  测试集1: {len(test1_ds):,}  测试集2: {len(test2_ds):,}')

# 快速验证：读一批看shape是否正确
xb, yb = next(iter(train_loader))
print(f'\n一批数据: X={xb.shape}  y={yb.shape}')
print(f'  H1={yb.sum().item()}  H0={(yb==0).sum().item()}')
print(f'  X值域: [{xb.min():.3f}, {xb.max():.3f}]')

✅ Dataset & DataLoader 构建完成
  训练集: 129,023  验证集: 32,257
  测试集1: 107,520  测试集2: 192,000

一批数据: X=torch.Size([16, 1024, 2])  y=torch.Size([16])
  H1=6  H0=10
  X值域: [-3.782, 3.932]


In [69]:
# ============================================================
# Cell A: 多模态特征构造函数
#
#   1. 时域：I/Q + 幅度 + 相位
#   2. 频域：STFT dB 功率谱
#   3. 关联域：GASF / GADF / RP
#
# 注意：
#   - 讲义中使用 pyts.image.GramianAngularField 和 RecurrencePlot
#   - 但你的 Kaggle 环境可能出现 No module named 'pyts'
#   - 所以这里采用“优先使用 pyts，不可用则自动使用公式手写版”的方式
# ============================================================

IMG_SIZE = 64

# ------------------------------------------------------------
# 2. 图像归一化函数
# ------------------------------------------------------------
def normalize_01(img):
    """
    将二维图像归一化到 [0, 1]。

    为什么要归一化？
    不同模态的数值范围不同：
        STFT 是 dB 功率谱；
        GASF/GADF 是三角函数矩阵；
        RP 是二值递归图。
    统一到 [0,1] 后，更适合 CNN 输入。
    """
    img = np.asarray(img, dtype=np.float32)

    min_v = img.min()
    max_v = img.max()

    if max_v - min_v < 1e-8:
        return np.zeros_like(img, dtype=np.float32)

    return ((img - min_v) / (max_v - min_v + 1e-8)).astype(np.float32)


# ------------------------------------------------------------
# 3. 二维图像缩放函数
# ------------------------------------------------------------
def resize_2d(img, size=64):
    """
    将任意二维图像缩放到 size × size。

    STFT 输出尺寸不一定正好是 64×64，
    所以需要统一尺寸，方便 CNN 批量输入。
    """
    img = np.asarray(img, dtype=np.float32)

    h, w = img.shape
    zoom_h = size / h
    zoom_w = size / w

    # order=1 表示双线性插值
    out = zoom(img, (zoom_h, zoom_w), order=1)

    # 防止浮点缩放造成尺寸略大
    out = out[:size, :size]

    # 防止尺寸略小，使用边缘值补齐
    if out.shape[0] < size or out.shape[1] < size:
        pad_h = size - out.shape[0]
        pad_w = size - out.shape[1]
        out = np.pad(out, ((0, pad_h), (0, pad_w)), mode="edge")

    return out.astype(np.float32)


# ------------------------------------------------------------
# 4. 一维序列降采样函数
# ------------------------------------------------------------
def downsample_1d(x, size=64):
    """
    将 1024 点序列压缩到 64 点。

    为什么要降采样？
    GASF/GADF/RP 都会生成 N×N 的二维矩阵。
    如果直接使用 1024 点，会生成 1024×1024 图像，计算量太大。
    """
    x = np.asarray(x, dtype=np.float32)
    idx = np.linspace(0, len(x) - 1, size).astype(np.int32)
    return x[idx]


# ------------------------------------------------------------
# 5. 构造时域输入：I/Q + 幅度 + 相位
# ------------------------------------------------------------
def make_time_features(x):
    """
    构造时域 BiLSTM 输入。

    输入：
        x: (1024, 2)
           第 0 列为 I
           第 1 列为 Q

    输出：
        feat: (1024, 4)
              [I, Q, amplitude, phase]

    对应讲义：
        使用 BiLSTM 提取相位和幅度的动态演变特征。
    """

    x = np.asarray(x, dtype=np.float32)

    I = x[:, 0]
    Q = x[:, 1]

    # 构造复数 IQ 信号
    complex_signal = I + 1j * Q

    # 瞬时幅度：A(t) = sqrt(I^2 + Q^2)
    amplitude = np.abs(complex_signal)

    # 瞬时相位：theta(t) = angle(I + jQ)
    # unwrap 可以减少相位在 -pi 和 pi 之间跳变带来的突变
    phase = np.unwrap(np.angle(complex_signal))

    # 拼接成每个时间步 4 个特征
    feat = np.stack([I, Q, amplitude, phase], axis=1).astype(np.float32)

    # 对每个特征通道做标准化，避免相位数值范围过大
    mean = feat.mean(axis=0, keepdims=True)
    std = feat.std(axis=0, keepdims=True) + 1e-6
    feat = (feat - mean) / std

    return feat.astype(np.float32)


# ------------------------------------------------------------
# 6. 构造 STFT dB 功率谱图
# ------------------------------------------------------------
def make_stft_img(x, size=64):
    """
    构造 STFT dB 功率谱图。

    输入：
        x: (1024, 2)

    输出：
        img: (64, 64)

    对应讲义：
        STFT 将一段信号分成多个短时间窗口，
        并对每个窗口做傅里叶变换，从而得到时频能量分布。
    """

    x = np.asarray(x, dtype=np.float32)

    I = x[:, 0]
    Q = x[:, 1]

    # 构造复数信号 I + jQ
    complex_signal = I + 1j * Q

    # 归一化采样率
    fs = 1.0

    # 计算 STFT
    # nperseg=64：每个窗口 64 个采样点
    # noverlap=32：相邻窗口重叠 32 点
    f, t, Zxx = signal.stft(
        complex_signal,
        fs=fs,
        nperseg=64,
        noverlap=32,
        boundary=None
    )

    # 幅度谱
    magnitude = np.abs(Zxx)

    # 功率谱
    power = magnitude ** 2

    # dB 功率谱，避免 log10(0)
    log_power = 10 * np.log10(power + 1e-10)

    # 归一化 + 缩放
    img = normalize_01(log_power)
    img = resize_2d(img, size)

    return img.astype(np.float32)


def make_relational_img(x, size=64, percentage=20):
    """
    构造关联域多通道图像。

    输入：
        x: (1024, 2)

    输出：
        rel_img: (3, 64, 64)
                 第 0 通道：GASF
                 第 1 通道：GADF
                 第 2 通道：RP

    【原理说明】
    1. 先由 IQ 信号构造幅度序列：
           amplitude = sqrt(I^2 + Q^2)

    2. 使用 GASF / GADF 将一维幅度序列转换为二维 Gramian Angular 图：
           GASF: G_ij = cos(phi_i + phi_j)
           GADF: G_ij = sin(phi_i - phi_j)

    3. 使用 RP 构造递归图：
           R_ij = Θ(epsilon - ||x_i - x_j||)

    4. 最后将三种关联域图像按通道堆叠：
           rel_img.shape = (3, 64, 64)

    对应讲义：
        GASF / GADF 捕捉时间序列的全局非线性相关性；
        RP 捕捉时间序列的递归结构和重复模式。
    """

    # 提取 I/Q 通道
    I = x[:, 0]
    Q = x[:, 1]

    # 构造幅度序列
    amplitude = np.sqrt(I ** 2 + Q ** 2).astype(np.float32)

    # 降采样到 size 个点
    # 这样 GASF/GADF/RP 输出就是 64×64，而不是 1024×1024
    amplitude = downsample_1d(amplitude, size)

    # pyts 要求输入格式为：
    #     (n_samples, n_timestamps)
    # 这里单条样本，所以 reshape 成 (1, 64)
    data_reshaped = amplitude.reshape(1, -1)

    # --------------------------------------------------------
    # 1. GASF：Gramian Angular Summation Field
    # --------------------------------------------------------
    gasf_transformer = GramianAngularField(
        image_size=size,
        method='summation'
    )

    img_gasf = gasf_transformer.fit_transform(data_reshaped)[0]

    # --------------------------------------------------------
    # 2. GADF：Gramian Angular Difference Field
    # --------------------------------------------------------
    gadf_transformer = GramianAngularField(
        image_size=size,
        method='difference'
    )

    img_gadf = gadf_transformer.fit_transform(data_reshaped)[0]

    # --------------------------------------------------------
    # 3. RP：Recurrence Plot
    # threshold='point' 表示按照距离点的分位数选阈值
    # percentage=20 表示选取距离较小的 20% 作为递归点
    # --------------------------------------------------------
    rp_transformer = RecurrencePlot(
        threshold='point',
        percentage=percentage
    )

    img_rp = rp_transformer.transform(data_reshaped)[0]

    # --------------------------------------------------------
    # 4. 归一化
    # GASF/GADF 数值范围和 RP 不完全一致，
    # 统一归一化后更适合 CNN 输入。
    # --------------------------------------------------------
    img_gasf = normalize_01(img_gasf)
    img_gadf = normalize_01(img_gadf)
    img_rp = img_rp.astype(np.float32)

    # 三个关联域图像按通道堆叠
    rel_img = np.stack(
        [img_gasf, img_gadf, img_rp],
        axis=0
    )

    return rel_img.astype(np.float32)

In [70]:
# ============================================================
# Cell B: 多模态 Dataset 包装器
#
# 目的：
#   在不改动原 SpectrumDataset 的基础上，
#   把一条 IQ 样本转换为三种模态输入。
#
# 输入：
#   原始 SpectrumDataset 返回 x=(1024,2), y
#
# 输出：
#   (x_time, x_stft, x_rel), y
# ============================================================

class MultiModalSpectrumDataset(Dataset):
    """
    多模态频谱感知 Dataset。

    base_dataset:
        已经构建好的 SpectrumDataset，
        例如 train_ds、val_ds、test1_ds、test2_ds。

    size:
        图像模态统一尺寸，默认 64×64。
    """

    def __init__(self, base_dataset, size=64, percentage=20):
        self.base_dataset = base_dataset
        self.size = size
        self.percentage = percentage

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        # 原始样本：x=(1024,2), y=0/1
        x, y = self.base_dataset[idx]

        # torch tensor 转 numpy，方便使用 scipy / numpy 处理
        if torch.is_tensor(x):
            x_np = x.numpy()
        else:
            x_np = np.asarray(x, dtype=np.float32)

        # 1. 时域模态：I/Q + 幅度 + 相位
        x_time = make_time_features(x_np)              # (1024, 4)

        # 2. 频域模态：STFT dB 功率谱
        x_stft = make_stft_img(x_np, self.size)        # (64, 64)
        x_stft = x_stft[None, :, :]                    # (1, 64, 64)

        # 3. 关联域模态：GASF + GADF + RP
        x_rel = make_relational_img(
            x_np,
            size=self.size,
            percentage=self.percentage
        )                                             # (3, 64, 64)

        return (
            torch.from_numpy(x_time).float(),
            torch.from_numpy(x_stft).float(),
            torch.from_numpy(x_rel).float()
        ), y


# ============================================================
# 构建多模态 DataLoader
# ============================================================

BATCH_SIZE_MM = 128

train_mm_ds = MultiModalSpectrumDataset(train_ds, size=IMG_SIZE, percentage=20)
val_mm_ds   = MultiModalSpectrumDataset(val_ds,   size=IMG_SIZE, percentage=20)
test1_mm_ds = MultiModalSpectrumDataset(test1_ds, size=IMG_SIZE, percentage=20)
test2_mm_ds = MultiModalSpectrumDataset(test2_ds, size=IMG_SIZE, percentage=20)

train_mm_loader = DataLoader(
    train_mm_ds,
    batch_size=BATCH_SIZE_MM,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_mm_loader = DataLoader(
    val_mm_ds,
    batch_size=BATCH_SIZE_MM,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test1_mm_loader = DataLoader(
    test1_mm_ds,
    batch_size=BATCH_SIZE_MM,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test2_mm_loader = DataLoader(
    test2_mm_ds,
    batch_size=BATCH_SIZE_MM,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

# 快速检查一批数据
(x_time_b, x_stft_b, x_rel_b), y_b = next(iter(train_mm_loader))

print("✅ 多模态 DataLoader 构建完成")
print("x_time:", x_time_b.shape)   # (B, 1024, 4)
print("x_stft:", x_stft_b.shape)   # (B, 1, 64, 64)
print("x_rel :", x_rel_b.shape)    # (B, 3, 64, 64)
print("y     :", y_b.shape)

✅ 多模态 DataLoader 构建完成
x_time: torch.Size([128, 1024, 4])
x_stft: torch.Size([128, 1, 64, 64])
x_rel : torch.Size([128, 3, 64, 64])
y     : torch.Size([128])


In [71]:
# ============================================================
# Cell C: 多头多模态频谱感知网络
#
# 三个分支：
#   1. TimeBranch：BiLSTM 处理时域序列
#   2. FreqBranch：CNN 处理 STFT 图
#   3. RelBranch ：CNN 处理 GASF/GADF/RP 图
#
# 融合方式：
#   concat    : 特征拼接
#   attention : 模态注意力动态加权
# ============================================================

class TimeBiLSTMBranch(nn.Module):
    """
    时域分支：BiLSTM。

    输入：
        x_time: (B, 1024, 4)

    每个时间步的 4 个特征：
        I, Q, amplitude, phase

    输出：
        time_feat: (B, emb_dim)
    """

    def __init__(self, input_size=4, hidden_size=64, emb_dim=128, dropout=0.3):
        super().__init__()

        self.bilstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        self.proj = nn.Sequential(
            nn.LayerNorm(hidden_size * 2),
            nn.Linear(hidden_size * 2, emb_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

    def forward(self, x_time):
        # x_time: (B, 1024, 4)
        out, _ = self.bilstm(x_time)

        # 取最后一个时间步的双向隐藏状态
        last = out[:, -1, :]          # (B, hidden_size*2)

        feat = self.proj(last)        # (B, emb_dim)

        return feat


class CNNBranch(nn.Module):
    """
    通用 CNN 图像分支。

    可用于：
        STFT 图像分支：in_channels=1
        关联域图像分支：in_channels=3
    """

    def __init__(self, in_channels, emb_dim=128, dropout=0.3):
        super().__init__()

        self.features = nn.Sequential(
            # 输入：(B, C, 64, 64)
            nn.Conv2d(in_channels, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),          # 64 → 32

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),          # 32 → 16

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),          # 16 → 8

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            # 自适应平均池化，输出固定为 1×1
            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, emb_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

    def forward(self, x_img):
        x = self.features(x_img)
        feat = self.proj(x)
        return feat


class MultiHeadSpectrumNet(nn.Module):
    """
    多头多模态频谱感知网络。

    输入：
        x_time: (B, 1024, 4)
        x_stft: (B, 1, 64, 64)
        x_rel : (B, 3, 64, 64)

    输出：
        logits: (B, 2)

    fusion:
        "concat"    ：特征拼接
        "attention" ：模态注意力加权
    """

    def __init__(self, emb_dim=128, num_classes=2, dropout=0.3, fusion="concat"):
        super().__init__()

        assert fusion in ["concat", "attention"]

        self.fusion = fusion

        # 三个模态分支
        self.time_branch = TimeBiLSTMBranch(
            input_size=4,
            hidden_size=64,
            emb_dim=emb_dim,
            dropout=dropout
        )

        self.freq_branch = CNNBranch(
            in_channels=1,
            emb_dim=emb_dim,
            dropout=dropout
        )

        self.rel_branch = CNNBranch(
            in_channels=3,
            emb_dim=emb_dim,
            dropout=dropout
        )

        # 特征拼接融合
        if fusion == "concat":
            self.classifier = nn.Sequential(
                #把 384 维融合特征压缩成 128 维
                nn.Linear(emb_dim * 3, 128),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(128, num_classes)
            )

        # 注意力融合
        else:
            # 对每个模态特征输出一个权重分数
            # attention 不直接拼接，而是先堆叠
            self.attn = nn.Sequential(
                nn.Linear(emb_dim, 64),
                #激活函数 -1_1
                nn.Tanh(),
                nn.Linear(64, 1)
            )
            '''
            在 attention 融合里，三个模态不是拼接，而是加权求和：
            fused = α_time × time_feat + α_freq × freq_feat + α_rel  × rel_feat
            因为三个模态都是 128 维，所以加权求和之后仍然是：
            fused.shape = (B, 128)
            '''
            self.classifier = nn.Sequential(
                nn.Linear(emb_dim, 128),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(128, num_classes)
            )

    def forward(self, x_time, x_stft, x_rel, return_attention=False):
        # 三个分支分别提取特征
        time_feat = self.time_branch(x_time)   # (B, emb_dim)
        freq_feat = self.freq_branch(x_stft)   # (B, emb_dim)
        rel_feat  = self.rel_branch(x_rel)     # (B, emb_dim)

        # ----------------------------------------------------
        # 方式一：特征拼接
        # ----------------------------------------------------
        if self.fusion == "concat":
            fused = torch.cat([time_feat, freq_feat, rel_feat], dim=1)
            logits = self.classifier(fused)

            if return_attention:
                return logits, None

            return logits

        # ----------------------------------------------------
        # 方式二：注意力加权
        # ----------------------------------------------------
        else:
            # stack 后形状：(B, 3, emb_dim)
            feats = torch.stack([time_feat, freq_feat, rel_feat], dim=1)

            # 对三个模态分别打分：(B, 3, 1)
            scores = self.attn(feats)

            # softmax 得到权重：(B, 3, 1)
            weights = torch.softmax(scores, dim=1)

            # 加权求和：(B, emb_dim)
            fused = torch.sum(feats * weights, dim=1)

            logits = self.classifier(fused)

            if return_attention:
                return logits, weights.squeeze(-1)

            return logits


# ============================================================
# 实例化模型
# ============================================================

model_mm = MultiHeadSpectrumNet(
    emb_dim=128,
    num_classes=2,
    dropout=0.3,
    fusion="concat"      # 先用 concat；后面可改成 "attention"
).to(DEVICE)

total_params = sum(p.numel() for p in model_mm.parameters())

print(model_mm)
print(f"\n✅ 多模态模型参数量: {total_params:,}")

MultiHeadSpectrumNet(
  (time_branch): TimeBiLSTMBranch(
    (bilstm): LSTM(4, 64, batch_first=True, bidirectional=True)
    (proj): Sequential(
      (0): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (1): Linear(in_features=128, out_features=128, bias=True)
      (2): ReLU()
      (3): Dropout(p=0.3, inplace=False)
    )
  )
  (freq_branch): CNNBranch(
    (features): Sequential(
      (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (4): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (5): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (6): ReLU()
      (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (8): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1

In [72]:
# ============================================================
# Cell D: 多模态训练循环
# ============================================================

#交叉熵损失函数。
criterion = nn.CrossEntropyLoss()
#优化器负责根据 loss 的反向传播结果，更新模型参数。
optimizer = optim.Adam(
    model_mm.parameters(),
    lr=5e-4,
    weight_decay=1e-4
)
#定义学习率调度器
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=3
)


def run_epoch_multimodal(model, loader, criterion, optimizer, device, train=True):
    """
    运行一个 epoch。

    train=True:
        训练模式，执行反向传播和参数更新

    train=False:
        验证模式，只前向推理，不更新参数
    """

    if train:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    total_correct = 0
    total_n = 0

    ctx = torch.enable_grad() if train else torch.no_grad()

    with ctx:
        for (x_time, x_stft, x_rel), yb in loader:

            # 将三种模态和标签放到 GPU / CPU
            x_time = x_time.to(device, non_blocking=True)
            x_stft = x_stft.to(device, non_blocking=True)
            x_rel  = x_rel.to(device, non_blocking=True)
            yb     = yb.to(device, non_blocking=True)

            if train:
                optimizer.zero_grad()

            # 前向传播
            logits = model(x_time, x_stft, x_rel)

            # 计算损失
            loss = criterion(logits, yb)

            if train:
                # 反向传播
                loss.backward()

                # 梯度裁剪，防止训练不稳定
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)

                # 参数更新
                optimizer.step()

            # 计算预测类别
            preds = logits.argmax(dim=1)

            # 累计统计
            total_correct += (preds == yb).sum().item()
            total_loss += loss.item() * len(yb)
            total_n += len(yb)

    avg_loss = total_loss / total_n
    avg_acc = total_correct / total_n

    return avg_loss, avg_acc


# ============================================================
# 正式训练
# ============================================================

NUM_EPOCHS = 3

history_mm = {
    "tr_loss": [],
    "val_loss": [],
    "tr_acc": [],
    "val_acc": []
}

best_val_loss = float("inf")
best_state = None
best_epoch = 0

print(f"🚀 开始训练多模态模型，共 {NUM_EPOCHS} epoch\n")
print(f'{"Ep":>4}|{"TrLoss":>9}|{"TrAcc":>8}|{"ValLoss":>9}|{"ValAcc":>8}|{"LR":>9}')
print("-" * 52)

for ep in range(1, NUM_EPOCHS + 1):

    tr_l, tr_a = run_epoch_multimodal(
        model_mm,
        train_mm_loader,
        criterion,
        optimizer,
        DEVICE,
        train=True
    )

    vl_l, vl_a = run_epoch_multimodal(
        model_mm,
        val_mm_loader,
        criterion,
        optimizer,
        DEVICE,
        train=False
    )

    scheduler.step(vl_l)

    history_mm["tr_loss"].append(tr_l)
    history_mm["val_loss"].append(vl_l)
    history_mm["tr_acc"].append(tr_a)
    history_mm["val_acc"].append(vl_a)

    if vl_l < best_val_loss:
        best_val_loss = vl_l
        best_epoch = ep
        best_state = {
            k: v.detach().cpu().clone()
            for k, v in model_mm.state_dict().items()
        }

    lr = optimizer.param_groups[0]["lr"]

    print(f'{ep:>4}|{tr_l:>9.4f}|{tr_a:>7.2%}|{vl_l:>9.4f}|{vl_a:>7.2%}|{lr:>9.2e}')

# 加载验证集最优模型
model_mm.load_state_dict(best_state)

print(f"\n🏆 最优模型出现在 Epoch {best_epoch}")
print(f"🏆 最优验证损失: {best_val_loss:.4f}")

🚀 开始训练多模态模型，共 3 epoch

  Ep|   TrLoss|   TrAcc|  ValLoss|  ValAcc|       LR
----------------------------------------------------


KeyboardInterrupt: 

In [ ]:
# ============================================================
# Cell E: 多模态模型三阶段评估
# ============================================================

def predict_multimodal_dataset(model, loader, device):
    """
    对整个数据集进行推理。

    输出：
        y_pred: 预测标签
        y_true: 真实标签
        y_prob: softmax 概率
    """

    model.eval()

    preds_all = []
    trues_all = []
    probs_all = []

    with torch.no_grad():
        for (x_time, x_stft, x_rel), yb in loader:

            x_time = x_time.to(device)
            x_stft = x_stft.to(device)
            x_rel  = x_rel.to(device)

            logits = model(x_time, x_stft, x_rel)
            probs = torch.softmax(logits, dim=1)

            preds_all.extend(logits.argmax(dim=1).cpu().numpy())
            trues_all.extend(yb.numpy())
            probs_all.extend(probs.cpu().numpy())

    return (
        np.array(preds_all),
        np.array(trues_all),
        np.array(probs_all)
    )


def full_metrics(y_true, y_pred, y_prob, name=""):
    """
    计算频谱感知任务核心指标。

    H0: 没有主用户
    H1: 有主用户

    Pd:
        检测概率，真实 H1 中被正确判为 H1 的比例。

    Pfa:
        虚警概率，真实 H0 中被错误判为 H1 的比例。
    """

    prob_h1 = y_prob[:, 1]

    cm = confusion_matrix(y_true, y_pred)
    TN, FP, FN, TP = cm.ravel()

    Pd = TP / (TP + FN + 1e-10)
    Pfa = FP / (FP + TN + 1e-10)

    fpr, tpr, _ = roc_curve(y_true, prob_h1)
    roc_auc = auc(fpr, tpr)

    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)

    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)
    print(f"Accuracy : {acc:.4f}")
    print(f"F1-score : {f1:.4f}")
    print(f"AUC      : {roc_auc:.4f}")
    print(f"Pd       : {Pd:.4f}")
    print(f"Pfa      : {Pfa:.4f}")
    print(f"Pmd      : {1 - Pd:.4f}")
    print("\nConfusion Matrix:")
    print(cm)
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, digits=4))

    return {
        "acc": acc,
        "f1": f1,
        "auc": roc_auc,
        "Pd": Pd,
        "Pfa": Pfa,
        "cm": cm,
        "fpr": fpr,
        "tpr": tpr,
        "prob_h1": prob_h1
    }


print("📊 开始多模态模型三阶段评估...")

pred_v_mm, true_v_mm, prob_v_mm = predict_multimodal_dataset(
    model_mm,
    val_mm_loader,
    DEVICE
)

pred_t1_mm, true_t1_mm, prob_t1_mm = predict_multimodal_dataset(
    model_mm,
    test1_mm_loader,
    DEVICE
)

pred_t2_mm, true_t2_mm, prob_t2_mm = predict_multimodal_dataset(
    model_mm,
    test2_mm_loader,
    DEVICE
)

m_v_mm = full_metrics(
    true_v_mm,
    pred_v_mm,
    prob_v_mm,
    "验证集 Validation：多模态融合模型"
)

m_t1_mm = full_metrics(
    true_t1_mm,
    pred_t1_mm,
    prob_t1_mm,
    "测试集1 Test1：SNR 泛化"
)

m_t2_mm = full_metrics(
    true_t2_mm,
    pred_t2_mm,
    prob_t2_mm,
    "测试集2 Test2：未知调制泛化"
)

print("\n" + "=" * 70)
print(f'{"指标":<10}{"验证集":>12}{"测试集1":>12}{"测试集2":>12}')
print("-" * 70)

for k, lab in [
    ("acc", "Accuracy"),
    ("f1", "F1"),
    ("auc", "AUC"),
    ("Pd", "Pd"),
    ("Pfa", "Pfa")
]:
    print(f'{lab:<10}{m_v_mm[k]:>12.4f}{m_t1_mm[k]:>12.4f}{m_t2_mm[k]:>12.4f}')

In [ ]:
# ============================================================
# Cell F: 训练曲线可视化
# ============================================================

eps = range(1, len(history_mm["tr_loss"]) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss 曲线
axes[0].plot(eps, history_mm["tr_loss"], marker="o", label="Train Loss")
axes[0].plot(eps, history_mm["val_loss"], marker="o", label="Val Loss")
axes[0].set_title("Multi-modal Loss Curve")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(alpha=0.3)

# Accuracy 曲线
axes[1].plot(eps, history_mm["tr_acc"], marker="o", label="Train Acc")
axes[1].plot(eps, history_mm["val_acc"], marker="o", label="Val Acc")
axes[1].set_title("Multi-modal Accuracy Curve")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Cell G: 按 SNR 统计 Pd
#
# 目的：
#   验证多模态融合是否真的提升了低信噪比下的检测概率。
# ============================================================

def pd_by_snr_for_h1(prob_h1_all, snr_h1, n_h1, threshold=0.5):
    """
    只针对 H1 样本计算不同 SNR 下的 Pd。

    参数：
        prob_h1_all:
            整个测试集的 H1 概率，shape = (N_total,)

        snr_h1:
            H1 样本对应的 SNR 数组

        n_h1:
            测试集中 H1 样本数量。
            注意：你的 Dataset 构造顺序是 H1 在前，H0 在后。

        threshold:
            判决阈值，默认 0.5。

    返回：
        每个 SNR 下的 Pd 字典。
    """

    # 取出前 n_h1 个样本，它们是 H1 样本
    prob_h1 = prob_h1_all[:n_h1]

    result = {}

    for s in sorted(np.unique(snr_h1)):
        mask = (snr_h1 == s)

        if mask.sum() == 0:
            continue

        preds_here = (prob_h1[mask] >= threshold).astype(int)

        # 对 H1 样本而言，预测为 1 的比例就是 Pd
        pd_here = preds_here.mean()

        result[s] = pd_here

    return result


# 测试集1：已知调制，训练外 SNR
pd_snr_t1_mm = pd_by_snr_for_h1(
    prob_h1_all=prob_t1_mm[:, 1],
    snr_h1=snr_t1_h1,
    n_h1=len(t1_h1_pos),
    threshold=0.5
)

# 测试集2：未知调制，全 SNR
pd_snr_t2_mm = pd_by_snr_for_h1(
    prob_h1_all=prob_t2_mm[:, 1],
    snr_h1=snr_t2_h1,
    n_h1=len(t2_h1_pos),
    threshold=0.5
)

print("\n📌 测试集1：按 SNR 统计 Pd")
for s, pd_value in pd_snr_t1_mm.items():
    print(f"SNR={s:>4.0f} dB | Pd={pd_value:.4f}")

print("\n📌 测试集2：按 SNR 统计 Pd")
for s, pd_value in pd_snr_t2_mm.items():
    print(f"SNR={s:>4.0f} dB | Pd={pd_value:.4f}")


# 可视化
plt.figure(figsize=(10, 5))

plt.plot(
    list(pd_snr_t1_mm.keys()),
    list(pd_snr_t1_mm.values()),
    marker="o",
    label="Test1: Known Mod, Unseen SNR"
)

plt.plot(
    list(pd_snr_t2_mm.keys()),
    list(pd_snr_t2_mm.values()),
    marker="s",
    label="Test2: Unknown Mod"
)

plt.axvspan(-20, -15, alpha=0.15, label="Very Low SNR Region")

plt.xlabel("SNR (dB)")
plt.ylabel("Detection Probability Pd")
plt.title("Pd vs SNR of Multi-modal Spectrum Sensing")
plt.grid(alpha=0.3)
plt.legend()
plt.show()